Scrape the content of https://www.lemonde.fr/Links to an external site. and save it as a CSV.

We want: titles, subhead, article URL, whether it's premium or not, byline, article type, image URL.

Bonus, if you want to get fancy:

Make the CSV file auto-updating. Use this tutorial (videoLinks to an external site., textLinks to an external site.) but just ignore the visualization/datawrapper aspect
Tips:

Use the 02 - BBC Scraping.ipynb notebook as a guide for this one. We used .get('href', None) in class to avoid "the link doesn't exist" problems, but it uses try/except. Both are reasonable, but try/except is more flexible.
You're going to be missing some article URLs - you should use try/except on them to avoid them being a problem, buuuut after that I would recommend printing the element to see why your approach isn't working (it's 100% possible to get all of the URLs!)
If you want to make sure the element you're getting has a specific attribute, you can do something like .select_one("a[href]"). That will only give you anchor tags (links) that have hrefs.
Instead of yes/no for the premium question, you can think of it as "put text in this column if the article is premium, don't put text in it if it is not"

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

response = requests.get("https://www.lemonde.fr/en/#")
doc = BeautifulSoup(response.text, 'html.parser')

In [2]:
items = doc.find_all('div', attrs={"class" : "article"})
len(items)

25

In [3]:
rows = []
for item in items:
    print('======='*20)
    row = {}
    
    headline = item.find(class_="article__title").text.strip()
    row['headline'] = headline
    print(headline)

    subhead = item.find(class_="article__desc")
    if subhead is not None:
        subhead = subhead.text.strip()
        row['subhead'] = subhead
        print(subhead)
    else:
        subhead = None
        row['subhead'] = np.nan
        print("no subhead")

    byline = item.find(class_="article__byline")
    if byline is not None:
        byline = byline.text.strip()
        row['byline'] = byline
        print(byline)
    else:
        byline = 'Unknown'
        row['byline'] = byline
        print(byline)
        
    url = item.find('a').get('href', None)
    row['url'] = url
    print(url)

    article_type = item.find(class_='article__type')
    if article_type is not None:
        article_type = article_type.text.strip()
        row['article_type'] = article_type
        print(article_type)
    else:
        article_type = None
        row['article_type'] = np.nan
        print("no article type")

    premium = item.find(class_='sr-only')
    if premium is not None:
        premium = True
        row['premium'] = premium
        print("Premium?", premium)
    else:
        premium = False
        row['premium'] = False
        print("Premium?", premium)

    image_url = item.find(class_='article__media')
    if image_url is not None:
        image_url = image_url.find('img').get('src', None)
        row['image_url'] = image_url
        print(image_url)
    else:
        image_url = None
        row['image_url'] = image_url
        print("No image")

    rows.append(row)

Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit
European leaders, who would like the US to reengage in the conflict between Moscow and Kyiv, encouraged the American president's enthusiasm about his memorandum of understanding with Tehran and confirmed their offer of assistance in the Strait of Hormuz.
Unknown
https://www.lemonde.fr/en/international/article/2026/06/17/trump-buoyed-by-his-deal-with-iran-shows-renewed-interest-in-ukraine-at-g7-summit_6754581_4.html
no article type
Premium? True
https://img.lemde.fr/2026/06/17/333/0/5000/3333/400/266/75/0/d639939_upload-1-9y1lmm3xjfyc-kzihnioglu-evian-g7-16062026-15.jpg
French government urged to reveal hidden assets of the rich
no subhead
Unknown
https://www.lemonde.fr/en/politics/article/2026/06/17/france-no-longer-knows-who-its-wealthiest-citizens-are-government-urged-to-reopen-black-box-on-the-assets-of-the-richest_6754601_5.html
no article type
Premium? True
https://img.lemde.fr/2025/09/19/0/0/1500/

In [4]:
# #trying again with try/except:

# rows = []
# for item in items[:3]:
#     print('======='*20)
#     row = {}
    
#     try:
#         row['headline'] = item.find(class_="article__title-label").text.strip()
#     except:
#         print("no headline found")

#     try:
#         row['subhead'] = item.find(class_="article__desc").text.strip()
#     except:
#         print("no subhead found")

#     try:
#         row['byline'] = item.find(class_="article__byline").text.strip()
#     except:
#         print("no byline found")

#     try:
#         row['url'] = item.find('a').get('href', None)
#     except:
#         print("no url found")
        
#     try:
#         row['article_type'] = item.find(class_='article__type').text.strip()
#     except:
#         print("no article_type found")   

#     try:
#         row['premium'] = item.find(class_='icon__premium')
#     except:
#         print("not premium") 

#     try:
#         row['image_url'] = item.find(class_='article__media').find('img').get('src', None)
#     except:
#         print("no image_url found") 

#     print(row)
#     rows.append(row)

try/except is not working but if/else is

In [5]:
rows

[{'headline': 'Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit',
  'subhead': "European leaders, who would like the US to reengage in the conflict between Moscow and Kyiv, encouraged the American president's enthusiasm about his memorandum of understanding with Tehran and confirmed their offer of assistance in the Strait of Hormuz.",
  'byline': 'Unknown',
  'url': 'https://www.lemonde.fr/en/international/article/2026/06/17/trump-buoyed-by-his-deal-with-iran-shows-renewed-interest-in-ukraine-at-g7-summit_6754581_4.html',
  'article_type': nan,
  'premium': True,
  'image_url': 'https://img.lemde.fr/2026/06/17/333/0/5000/3333/400/266/75/0/d639939_upload-1-9y1lmm3xjfyc-kzihnioglu-evian-g7-16062026-15.jpg'},
 {'headline': 'French government urged to reveal hidden assets of the rich',
  'subhead': nan,
  'byline': 'Unknown',
  'url': 'https://www.lemonde.fr/en/politics/article/2026/06/17/france-no-longer-knows-who-its-wealthiest-citizens-are-government-u

In [6]:
df = pd.json_normalize(rows)
df.head()

,headline,subhead,byline,url,article_type,premium,image_url
0,"Trump, buoyed by his deal with Iran, shows ren...","European leaders, who would like the US to ree...",Unknown,https://www.lemonde.fr/en/international/articl...,NaN,True,https://img.lemde.fr/2026/06/17/333/0/5000/333...
1,French government urged to reveal hidden asset...,NaN,Unknown,https://www.lemonde.fr/en/politics/article/202...,NaN,True,https://img.lemde.fr/2025/09/19/0/0/1500/1000/...
2,French security replaces Palantir with domesti...,NaN,Unknown,https://www.lemonde.fr/en/france/article/2026/...,NaN,True,https://img.lemde.fr/2026/06/11/0/0/6000/4000/...
3,2026 World Cup: International press hails long...,With outstanding performances in their opening...,Unknown,https://www.lemonde.fr/en/sports/article/2026/...,NaN,True,https://img.lemde.fr/2026/06/16/0/92/5283/3522...
4,'Japan's PM hopes to boost growth through publ...,"The Bank of Japan, the last major central bank...",Unknown,https://www.lemonde.fr/en/economy/article/2026...,NaN,True,https://img.lemde.fr/2026/06/17/0/0/7782/5188/...


In [7]:
df.to_csv('lemonde_fr_news.csv', index=False)